# 🏭 VELARA GPU Factory — Colab Worker

**Kya karna hai (sirf 2 steps):**
1. `Runtime ▸ Run all` dabao (ya neeche cell ke ▶ button).
2. Jab **`ssh ....tmate.io`** wali line print ho jaye, usse **copy karke chat mein assistant ko paste** kar do.

Bas! Assistant ab Colab ke **T4 GPU** se Blender ke heavy jobs (2048px bakes, high-sample renders)
chala lega — 10-50× faster than codespace CPU.

⚠️ **Tab khula rakho** — tab/Colab band hua to worker mar jayega (~12 ghante max session).
GPU nahi mila to bhi chalega (CPU fallback).


In [ ]:
# ============================================================
#  VELARA GPU FACTORY — Colab worker setup (ek hi cell)
#  Output: tmate ssh-line -> assistant ko paste karo
# ============================================================
import subprocess, os, time

def run(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[:400])
    if r.returncode != 0 and r.stderr.strip():
        print('ERR:', r.stderr.strip()[:300])
    return r.returncode

print('== 1) GPU check ==')
g = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv',
                   shell=True, capture_output=True, text=True)
print(g.stdout.strip() if g.returncode == 0
      else 'NO GPU — Runtime > Change runtime type > T4 GPU, phir Run All')

print('\n== 2) deps + Blender 4.5.16 (headless) ==')
run('apt-get -qq update')
run('apt-get -qq install -y libx11-6 libxi6 libxxf86vm1 libxfixes3 libxrender1 '
    'libgl1 libxkbcommon0 libsm6 libice6 xz-utils tmate')
if not os.path.exists('/opt/blender/blender'):
    run('curl -fsSL -o /tmp/b.tar.xz '
        'https://download.blender.org/release/Blender4.5/blender-4.5.16-linux-x64.tar.xz')
    run('mkdir -p /opt/blender && tar -xJf /tmp/b.tar.xz -C /opt/blender '
        '--strip-components=1 && rm /tmp/b.tar.xz')
v = subprocess.run('/opt/blender/blender --version', shell=True,
                   capture_output=True, text=True)
print(v.stdout.splitlines()[0] if v.returncode == 0 and v.stdout else 'BLENDER INSTALL FAIL')

print('\n== 3) repo clone (public repo hai) ==')
if not os.path.exists('/content/Velara'):
    rc = run('git clone --depth 1 https://github.com/priyanshagrahari54-blip/Velara.git /content/Velara')
    if rc != 0:
        print('(clone skip — assistant files ssh se bhejega)')

print('\n== 4) tmate bridge (assistant ka raasta) ==')
run('tmate kill-server 2>/dev/null || true')
run('tmate new-session -d')
time.sleep(6)
line = subprocess.run("tmate display -p '#{tmate_ssh}'", shell=True,
                      capture_output=True, text=True).stdout.strip()

print('=' * 62)
print('YE LINE ASSISTANT KO CHAT MEIN PASTE KARO:')
print('    ' + line)
print('=' * 62)
print('Notebook ka tab khula rakho. Worker ~12 ghante tak ya disconnect hone tak zinda.')
